# Audiobook Engine: Stage 2, pronunciation auditions

Generate a small, reversible listening set from the approved 13-term pronunciation inventory. This stage uses **Chatterbox Multilingual V3** in English and Hindi. It compares the book's spelling, an IAST transliteration where different, and a Devanagari Hindi rendering. It never edits the approved manuscript or approves any pronunciation automatically.

**Use a Colab GPU runtime.** The model is downloaded from Hugging Face and cached in Google Drive so the checkpoint can survive a runtime reset. No paid API is used.

## 1. Connect Drive and refresh the project

This stage consumes the approved files produced by Stage 1. It does not re-extract the PDF or rerun the repair stage.

In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request, json, shutil, zipfile, base64, hashlib, csv
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

REPOSITORY = 'https://github.com/privacyfreak84/colabchatterLLMthingstuff.git'
PROJECT_DIR = Path('/content/colabchatterLLMthingstuff')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', 'main:refs/remotes/origin/main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '-B', 'main', 'origin/main'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git repository.')

OUTPUT_DIR = Path('/content/drive/MyDrive/Audiobook/Buddhist Yoga/reconnaissance')
required = [
    'main_manuscript_approved.txt',
    'notes_manuscript_approved.txt',
    'manuscript_approval_record.json',
    'pronunciation_audition_sheet.csv',
]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing approved Stage 1 files: {missing}')

sys.path.insert(0, str(PROJECT_DIR / 'tools'))

# Persist Hugging Face downloads in Drive rather than the ephemeral Colab VM.
HF_HOME = Path('/content/drive/MyDrive/Audiobook/.cache/huggingface')
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')

run_stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
AUDITION_DIR = OUTPUT_DIR / 'pronunciation_auditions' / f'v3-{run_stamp}'
AUDITION_DIR.mkdir(parents=True, exist_ok=False)
print('Approved source files found. Output folder:', AUDITION_DIR)

## 2. Install the Chatterbox runtime without replacing Colab's Torch

The current Chatterbox package pins particular Torch versions in its dependency metadata. This cell installs the official package without dependencies, then installs the inference libraries separately, leaving Colab's preinstalled `torch` and `torchaudio` in place. If an import check fails, stop rather than silently changing the GPU runtime.

In [ ]:
package_specs = [
    'librosa==0.11.0',
    's3tokenizer',
    'transformers==5.2.0',
    'diffusers==0.29.0',
    'git+https://github.com/resemble-ai/Perth.git@master',
    'conformer==0.3.2',
    'safetensors==0.5.3',
    'omegaconf',
    'soundfile',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'chatterbox-tts==0.1.7'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *package_specs],
    check=True,
)

import numpy as np
import torch
import torchaudio
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, '| torchaudio:', torchaudio.__version__)
print('NumPy:', np.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun this notebook.')
if torch.__version__.split('+')[0].split('.')[:2] != torchaudio.__version__.split('+')[0].split('.')[:2]:
    raise RuntimeError('Torch and torchaudio versions do not match. Do not continue with an inconsistent audio runtime.')
if sys.version_info < (3, 13) and int(np.__version__.split('.')[0]) >= 2:
    raise RuntimeError('This Chatterbox release specifies NumPy <2 for Python <3.13. Install numpy<2, restart the runtime, and rerun Stage 2.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 1024**3, 1))

## 3. Download the language-matched audition references and load V3

The English and Hindi reference clips are public demo samples from Resemble AI. They are used only to make the pronunciation tests language-matched. They are **not** the final audiobook voice. We will choose the narration voice separately.

In [ ]:
EN_REF = AUDITION_DIR / 'official_demo_en_f1.flac'
HI_REF = AUDITION_DIR / 'official_demo_hi_f1.flac'
reference_urls = {
    EN_REF: 'https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac',
    HI_REF: 'https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac',
}
for path, url in reference_urls.items():
    if not path.is_file():
        urllib.request.urlretrieve(url, path)
    if path.stat().st_size < 1024:
        raise RuntimeError(f'Reference audio download appears incomplete: {path}')

from chatterbox.mtl_tts import ChatterboxMultilingualTTS, punc_norm
supported = ChatterboxMultilingualTTS.get_supported_languages()
if not {'en', 'hi'}.issubset(supported):
    raise RuntimeError(f'The installed Chatterbox package does not expose both English and Hindi: {supported}')

print('Loading Chatterbox Multilingual V3. First download may be several GB; it is cached in Drive.')
model = ChatterboxMultilingualTTS.from_pretrained(device='cuda', t3_model='v3')
print('Loaded V3. Sample rate:', model.sr, '| languages:', supported.get('en'), '/', supported.get('hi'))

## 4. Tokenizer preflight and pronunciation auditions

For every term, the helper creates up to three samples:

1. **Source English:** the original book spelling inside its source sentence.
2. **IAST English:** the same source sentence with a standard Sanskrit transliteration substituted, where the spelling differs.
3. **Devanagari Hindi:** the Sanskrit term rendered in Devanagari inside a short Hindi carrier sentence.

The tokenizer is checked before each sample. Inputs containing `[UNK]` tokens are skipped and recorded, rather than silently synthesizing dropped characters. Paired English variants use the same seed. The Hindi samples use a Hindi reference clip and are clearly labelled as a separate-language audition.

In [ ]:
from pronunciation_audition import generate_audition_set

audition_summary = generate_audition_set(
    model=model,
    audition_sheet=OUTPUT_DIR / 'pronunciation_audition_sheet.csv',
    output_dir=AUDITION_DIR,
    english_reference_audio=EN_REF,
    hindi_reference_audio=HI_REF,
    punc_norm=punc_norm,
    seed_base=20261030,
)
print(json.dumps(audition_summary, ensure_ascii=False, indent=2))
if audition_summary['generated_count'] == 0:
    raise RuntimeError('No audition audio was generated. Inspect pronunciation_audition_tasks.csv before continuing.')

## 5. Listen and record decisions

Listen to each comparison before changing the decision sheet. The Devanagari Hindi sample uses a different language reference voice; it is a pronunciation control, not a direct voice-quality comparison with the English samples. No variant is preselected.

In [ ]:
from IPython.display import Audio, Markdown, display

decision_csv = AUDITION_DIR / 'pronunciation_audition_decisions.csv'
with decision_csv.open('r', encoding='utf-8-sig', newline='') as handle:
    decision_rows = list(csv.DictReader(handle))

audio_columns = [
    ('Source spelling, English V3', 'source_english_audio'),
    ('IAST spelling, English V3', 'iast_english_audio'),
    ('Devanagari, Hindi V3', 'devanagari_hindi_audio'),
]
for row in decision_rows:
    display(Markdown(f"### {row['audition_id']}: {row['source_term']}  ({row['occurrences']} occurrences)"))
    for label, field in audio_columns:
        filename = row.get(field, '')
        if filename:
            display(Markdown(f'**{label}**'))
            display(Audio(filename=str(AUDITION_DIR / filename), autoplay=False))
        elif field == 'iast_english_audio':
            display(Markdown('_IAST spelling is identical to the source spelling; duplicate audio was skipped._'))
        else:
            display(Markdown(f'_No sample generated for {label}. Check the task manifest._'))

## 6. Package the audition set for review

The ZIP contains the WAVs, tokenizer preflight results, task manifest, and blank decision template. A Base64 text wrapper is also created so it can be uploaded to the project repository using the same transfer method as the earlier reconnaissance bundles. This bundle is still only an audition set.

In [ ]:
zip_path = OUTPUT_DIR / 'pronunciation_auditions_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(AUDITION_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(AUDITION_DIR))

encoded_path = OUTPUT_DIR / 'pronunciation_auditions_bundle.txt'
encoded_path.write_text(base64.b64encode(zip_path.read_bytes()).decode('ascii') + '\n', encoding='ascii')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print(json.dumps({
    'zip_path': str(zip_path),
    'zip_bytes': zip_path.stat().st_size,
    'zip_sha256': sha256(zip_path),
    'base64_transfer_file': str(encoded_path),
    'base64_chars': encoded_path.stat().st_size,
    'generated_audio_files': audition_summary['generated_count'],
    'full_book_synthesis_authorized': False,
}, ensure_ascii=False, indent=2))
print('\nUpload pronunciation_auditions_bundle.txt to the repository as nextrecon2.txt, then tell ChatGPT it is uploaded.')

## Stop here

Do not synthesize the audiobook yet. We will inspect the audition bundle, compare the actual samples, select a spoken form for each term, and only then create an approved pronunciation dictionary. The original manuscript spelling remains unchanged unless a specific reversible narration substitution is later approved.